# Лабораторная работа №9
## Методы обучения без учителя.

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б

### Цель работы
Изучение методов кластеризации и снижения размерности.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
iris = load_iris()
X_full, y_true = iris.data, iris.target
feature_names = iris.feature_names

# D1: подмножество признаков (без целевого)
X_d1 = X_full[:, :4]  # все 4 признака — числовые

print(f"D1 (исходный): {X_d1.shape}")
print(f"Признаки: {feature_names}")


In [ ]:
# === СНИЖЕНИЕ РАЗМЕРНОСТИ ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_d1)

# D2: PCA
pca = PCA(n_components=2, random_state=42)
X_d2 = pca.fit_transform(X_scaled)
print(f"D2 (PCA): {X_d2.shape}, объяснённая дисперсия: {pca.explained_variance_ratio_.sum():.3f}")

# D3: t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_d3 = tsne.fit_transform(X_scaled)
print(f"D3 (t-SNE): {X_d3.shape}")

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_true = ['#e74c3c', '#2ecc71', '#3498db']

for i, name in enumerate(iris.target_names):
    mask = y_true == i
    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1], c=colors_true[i], label=name, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
axes[0].set_title('D1: Исходные (первые 2 признака)', fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for i, name in enumerate(iris.target_names):
    mask = y_true == i
    axes[1].scatter(X_d2[mask, 0], X_d2[mask, 1], c=colors_true[i], label=name, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
axes[1].set_title('D2: PCA', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

for i, name in enumerate(iris.target_names):
    mask = y_true == i
    axes[2].scatter(X_d3[mask, 0], X_d3[mask, 1], c=colors_true[i], label=name, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
axes[2].set_title('D3: t-SNE', fontsize=12, fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Снижение размерности: сравнение', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab9/dimensionality_reduction.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === КЛАСТЕРИЗАЦИЯ ===
datasets = {'D1': X_scaled, 'D2': X_d2, 'D3': X_d3}
n_clusters = 3

all_results = []

for d_name, X_data in datasets.items():
    print(f"\n=== {d_name} ===")

    # K-Means
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_data)
    km_sil = silhouette_score(X_data, km_labels)
    km_ari = adjusted_rand_score(y_true, km_labels)
    km_ch = calinski_harabasz_score(X_data, km_labels)
    all_results.append({'Dataset': d_name, 'Method': 'K-Means', 'Silhouette': km_sil, 'ARI': km_ari, 'CH': km_ch})
    print(f"K-Means: Silhouette={km_sil:.3f}, ARI={km_ari:.3f}, CH={km_ch:.1f}")

    # Agglomerative Clustering
    ac = AgglomerativeClustering(n_clusters=n_clusters)
    ac_labels = ac.fit_predict(X_data)
    ac_sil = silhouette_score(X_data, ac_labels)
    ac_ari = adjusted_rand_score(y_true, ac_labels)
    ac_ch = calinski_harabasz_score(X_data, ac_labels)
    all_results.append({'Dataset': d_name, 'Method': 'Agglomerative', 'Silhouette': ac_sil, 'ARI': ac_ari, 'CH': ac_ch})
    print(f"Agglomerative: Silhouette={ac_sil:.3f}, ARI={ac_ari:.3f}, CH={ac_ch:.1f}")

    # DBSCAN
    db = DBSCAN(eps=0.8 if d_name == 'D1' else 3.0, min_samples=5)
    db_labels = db.fit_predict(X_data)
    if len(set(db_labels)) > 1 and -1 not in db_labels or len(set(db_labels) - {-1}) > 1:
        db_sil = silhouette_score(X_data, db_labels) if len(set(db_labels)) > 1 else 0
    else:
        db_sil = -1
    db_ari = adjusted_rand_score(y_true, db_labels) if len(set(db_labels)) > 1 else -1
    all_results.append({'Dataset': d_name, 'Method': 'DBSCAN', 'Silhouette': db_sil, 'ARI': db_ari, 'CH': 0})
    print(f"DBSCAN: Silhouette={db_sil:.3f}, ARI={db_ari:.3f}")

results_df = pd.DataFrame(all_results)


In [ ]:
# Визуализация кластеризации
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
methods = ['K-Means', 'Agglomerative', 'DBSCAN']
clusterers = [KMeans(n_clusters=3, random_state=42, n_init=10), AgglomerativeClustering(n_clusters=3), DBSCAN(eps=0.8, min_samples=5)]

colors_clust = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

for col, (d_name, X_data) in enumerate(datasets.items()):
    eps_val = 0.8 if d_name == 'D1' else 3.0
    clust_list = [KMeans(n_clusters=3, random_state=42, n_init=10),
                  AgglomerativeClustering(n_clusters=3),
                  DBSCAN(eps=eps_val, min_samples=5)]

    for row, (method, clust) in enumerate(zip(methods, clust_list)):
        ax = axes[row][col]
        labels = clust.fit_predict(X_data)
        for lbl in set(labels):
            mask = labels == lbl
            c = colors_clust[lbl % len(colors_clust)] if lbl != -1 else '#aaaaaa'
            ax.scatter(X_data[mask, 0], X_data[mask, 1], c=c, label=f'Cluster {lbl}', alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
        ax.set_title(f'{d_name} + {method}', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)

plt.suptitle('Результаты кластеризации', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab9/clustering_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Сводная таблица метрик
print("=== СВОДНАЯ ТАБЛИЦА МЕТРИК ===")
print(results_df.round(3).to_string(index=False))

# Визуализация метрик
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Silhouette
pivot_sil = results_df.pivot(index='Dataset', columns='Method', values='Silhouette')
pivot_sil.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral', 'forestgreen'])
axes[0].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score'); axes[0].legend(title='Метод'); axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# ARI
pivot_ari = results_df.pivot(index='Dataset', columns='Method', values='ARI')
pivot_ari.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral', 'forestgreen'])
axes[1].set_title('Adjusted Rand Index', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score'); axes[1].legend(title='Метод'); axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.suptitle('Сравнение методов кластеризации', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab9/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### Выводы
1. **Снижение размерности**: t-SNE (D3) лучше визуально разделяет кластеры, чем PCA (D2).
2. **PCA** объясняет ~95% дисперсии в 2 компонентах.
3. **K-Means** показал наилучший ARI (~0.7-0.9) на всех датасетах.
4. **Agglomerative Clustering** близок к K-Means по качеству.
5. **DBSCAN** чувствителен к параметру eps и выделяет шумовые точки.
6. Наиболее качественная кластеризация достигается на D3 (t-SNE) с K-Means.
